<a href="https://colab.research.google.com/github/moeenessa31-lgtm/Final-Project/blob/main/Credit_Card_Fraud_Detection_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mlg_ulb_creditcardfraud_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')

print('Data source import complete.')


# Credit Card Fraud Detection: Unsupervised Anomaly Detection and Deep Learning

**Project:** Anomaly Detection and Customer Segmentation Using Unsupervised Machine Learning and Deep Learning  
**Dataset:** Credit Card Fraud Detection Dataset  

**Team Members**  
- A'SEM ZAKARIYA ALI ELFRIEH — Student ID: 180293  
- HAMZA MOEEN RASHEED ESSA — Student ID: 183174
- MOHAMMAD ZIAD ABDELMAJEED AL ROUSAN — Student ID: 180601

This notebook contains the cleaned final implementation. Exploratory failed attempts, repeated trial plots, and unnecessary cells were removed. Tuning tables are kept only where they justify the selected final parameters.

## 1. Imports and Dataset Loading

The dataset is loaded from the Kaggle path used in the project environment. The `Class` column is kept only for evaluation: `0` means genuine transaction and `1` means fraud.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    roc_curve, auc, precision_recall_curve,
    silhouette_score, davies_bouldin_score
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Download latest version
import kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(path + "/creditcard.csv")
print("Original shape:", df.shape)
df.head()

## 2. Initial Data Exploration

This section checks the dataset structure, class imbalance, missing values, and duplicated rows.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Class counts:")
print(df["Class"].value_counts())
print("Class percentages:")
print(df["Class"].value_counts(normalize=True) * 100)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate rows by class:")
print(df[df.duplicated()]["Class"].value_counts())

## 3. Data Cleaning and Scaling

Duplicate rows are removed. RobustScaler is used because anomaly detection is sensitive to feature scale, and transaction data may contain extreme values. `Class` is not included in the model input.

In [ ]:
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")
print("Cleaned shape:", df.shape)

X = df.drop("Class", axis=1)
y = df["Class"]

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaled shape:", X_scaled.shape)
X_scaled.head()

## 4. PCA Dimensionality Reduction

PCA is applied after scaling. The number of components is selected using cumulative explained variance. A 95% threshold is used as a practical balance between information preservation and dimensionality reduction.

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

pca_variance_table = pd.DataFrame({
    "Principal Component": [f"PC{i+1}" for i in range(len(explained_variance))],
    "Explained Variance Ratio": explained_variance,
    "Cumulative Variance": cumulative_variance
})
pca_variance_table.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(y=0.95, linestyle="--", label="95% variance threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Cumulative Explained Variance")
plt.legend()
plt.grid(True)
plt.show()

n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print("Number of components needed for 95% variance:", n_components_95)
print("Cumulative variance at selected components:", cumulative_variance[n_components_95 - 1])

In [ ]:
pca_95 = PCA(n_components=n_components_95)
X_pca_95 = pca_95.fit_transform(X_scaled)
X_pca_95 = pd.DataFrame(X_pca_95, columns=[f"PC{i+1}" for i in range(n_components_95)])
print("PCA-reduced shape:", X_pca_95.shape)
X_pca_95.head()

In [ ]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca_2, columns=["PC1", "PC2"])
pca_df["Class"] = y.values

normal_pca = pca_df[pca_df["Class"] == 0]
fraud_pca = pca_df[pca_df["Class"] == 1]
pca_sample = pd.concat([normal_pca, fraud_pca])

plt.figure(figsize=(10, 6))

plt.scatter(pca_sample[pca_sample["Class"] == 0]["PC1"],
            pca_sample[pca_sample["Class"] == 0]["PC2"],
            alpha=0.4, s=10, label="Normal")

plt.scatter(pca_sample[pca_sample["Class"] == 1]["PC1"],
            pca_sample[pca_sample["Class"] == 1]["PC2"],
            alpha=0.9, s=2, label="Fraud")

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA 2D Visualization: Normal vs Fraud")
plt.legend()
plt.grid(True)
plt.show()

print("PC1 + PC2 cumulative variance:", cumulative_variance[1])

## 5. t-SNE Visualization

Full-data t-SNE was computationally expensive for more than 283,000 cleaned transactions. Therefore, a representative visualization subset is used: all fraud transactions plus 10,000 randomly selected normal transactions. Labels are used only for coloring, not for fitting t-SNE.

In [ ]:
pca_with_class = X_pca_95.copy()
pca_with_class["Class"] = y.values

fraud_data = pca_with_class[pca_with_class["Class"] == 1]
normal_data = pca_with_class[pca_with_class["Class"] == 0].sample(n=10000, random_state=RANDOM_STATE)

tsne_subset = pd.concat([normal_data, fraud_data], axis=0)
tsne_subset = tsne_subset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_tsne_subset = tsne_subset.drop("Class", axis=1)
y_tsne_subset = tsne_subset["Class"]

print("t-SNE subset shape:", X_tsne_subset.shape)
print("Class distribution:")
print(y_tsne_subset.value_counts())

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE,
    verbose=1
)

X_tsne = tsne.fit_transform(X_tsne_subset)

tsne_df = pd.DataFrame(X_tsne, columns=["TSNE1", "TSNE2"])
tsne_df["Class"] = y_tsne_subset.values

plt.figure(figsize=(10, 6))

plt.scatter(tsne_df[tsne_df["Class"] == 0]["TSNE1"],
            tsne_df[tsne_df["Class"] == 0]["TSNE2"],
            alpha=0.4, s=10, label="Normal")

plt.scatter(tsne_df[tsne_df["Class"] == 1]["TSNE1"],
            tsne_df[tsne_df["Class"] == 1]["TSNE2"],
            alpha=0.9, s=5, label="Fraud")

plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.title("t-SNE Visualization: Normal vs Fraud")
plt.legend()
plt.grid(True)
plt.show()

## 6. DBSCAN Clustering and Noise Detection

DBSCAN is used on the PCA-reduced t-SNE subset features (`X_tsne_subset`), not directly on t-SNE coordinates. The t-SNE coordinates are used only for visualization. Parameter tuning compares anomaly detection metrics and clustering quality metrics.

In [ ]:
min_samples_for_plot = 10
neighbors = NearestNeighbors(n_neighbors=min_samples_for_plot)
neighbors_fit = neighbors.fit(X_tsne_subset)
distances, indices = neighbors_fit.kneighbors(X_tsne_subset)
k_distances = np.sort(distances[:, min_samples_for_plot - 1])

print("k-distance percentiles:")
for p in [50, 75, 90, 95, 97, 98, 99, 99.5, 99.9]:
    print(f"{p}% percentile:", np.percentile(k_distances, p))

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.ylim(0, 20)
plt.xlabel("Points sorted by distance")
plt.ylabel(f"{min_samples_for_plot}th nearest-neighbor distance")
plt.title("Zoomed k-Distance Plot for Choosing DBSCAN eps")
plt.grid(True)
plt.show()

In [ ]:
min_samples_values = [5, 10, 20]
eps_values = [3, 4, 5, 6, 7, 8, 9, 10, 12]

dbscan_tuning_rows = []

for min_s in min_samples_values:
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_s, metric="euclidean", n_jobs=-1)
        labels = dbscan.fit_predict(X_tsne_subset)
        y_true = y_tsne_subset.values
        y_pred = (labels == -1).astype(int)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        noise_percentage = (n_noise / len(labels)) * 100
        fraud_detected = np.sum((y_true == 1) & (y_pred == 1))

        clustered_mask = labels != -1
        X_clustered = X_tsne_subset[clustered_mask]
        labels_clustered = labels[clustered_mask]
        if len(set(labels_clustered)) > 1:
            sil = silhouette_score(X_clustered, labels_clustered)
            dbi = davies_bouldin_score(X_clustered, labels_clustered)
        else:
            sil = np.nan
            dbi = np.nan

        dbscan_tuning_rows.append({
            "eps": eps,
            "min_samples": min_s,
            "clusters": n_clusters,
            "noise_points": n_noise,
            "noise_percentage": noise_percentage,
            "fraud_detected": fraud_detected,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1_score": f1_score(y_true, y_pred, zero_division=0),
            "silhouette_score": sil,
            "davies_bouldin_index": dbi
        })

dbscan_tuning_results = pd.DataFrame(dbscan_tuning_rows)
dbscan_tuning_results.sort_values(by="f1_score", ascending=False).head(10)

In [ ]:
final_eps = 6
final_min_samples = 20

final_dbscan = DBSCAN(eps=final_eps, min_samples=final_min_samples, metric="euclidean", n_jobs=-1)
final_dbscan_labels = final_dbscan.fit_predict(X_tsne_subset)

final_dbscan_results = pd.DataFrame({
    "Actual_Class": y_tsne_subset.values,
    "DBSCAN_Label": final_dbscan_labels
})
final_dbscan_results["Predicted_Anomaly"] = (final_dbscan_results["DBSCAN_Label"] == -1).astype(int)

print("Final Tuned DBSCAN")
print("eps:", final_eps)
print("min_samples:", final_min_samples)
print("Cluster distribution:")
print(pd.Series(final_dbscan_labels).value_counts().sort_index())
print("Confusion Matrix:")
print(confusion_matrix(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"]))
print("Classification Report:")
print(classification_report(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"], target_names=["Normal", "Fraud"]))

clustered_mask = final_dbscan_labels != -1
X_clustered = X_tsne_subset[clustered_mask]
labels_clustered = final_dbscan_labels[clustered_mask]
print("Silhouette Score:", silhouette_score(X_clustered, labels_clustered))
print("Davies-Bouldin Index:", davies_bouldin_score(X_clustered, labels_clustered))

In [ ]:
tsne_df["Final_DBSCAN_Label"] = final_dbscan_labels
tsne_df["Final_DBSCAN_Anomaly"] = (final_dbscan_labels == -1).astype(int)

plt.figure(figsize=(10, 6))

plt.scatter(tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 0]["TSNE1"],
            tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 0]["TSNE2"],
            alpha=0.4, s=10, label="Clustered Points")

plt.scatter(tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 1]["TSNE1"],
            tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 1]["TSNE2"],
            alpha=0.9, s=25, label="DBSCAN Noise / Anomaly")

plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.title("Final Tuned DBSCAN Noise Points on t-SNE")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))

plt.scatter(tsne_df[tsne_df["Class"] == 0]["TSNE1"],
            tsne_df[tsne_df["Class"] == 0]["TSNE2"],
            alpha=0.25, s=8, label="Actual Normal")

plt.scatter(tsne_df[tsne_df["Class"] == 1]["TSNE1"],
            tsne_df[tsne_df["Class"] == 1]["TSNE2"],
            alpha=0.9, s=25, label="Actual Fraud")

plt.scatter(tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 1]["TSNE1"],
            tsne_df[tsne_df["Final_DBSCAN_Anomaly"] == 1]["TSNE2"],
            facecolors="none", edgecolors="black", s=70, label="Final DBSCAN Detected Anomaly")

plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.title("Actual Fraud vs Final Tuned DBSCAN Detected Anomalies")
plt.legend()
plt.grid(True)
plt.show()

## 7. One-Class SVM Anomaly Detection

One-Class SVM is trained only on normal transactions. A normal training subset is used for computational efficiency, while evaluation is performed on the full test set.

In [ ]:
X_ocsvm = X_pca_95.reset_index(drop=True)
y_ocsvm = y.reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_ocsvm,
    y_ocsvm,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_ocsvm
)

X_train_normal = X_train[y_train == 0]
X_train_normal_sample = X_train_normal.sample(n=20000, random_state=RANDOM_STATE)

print("Full data shape:", X_ocsvm.shape)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Normal training sample shape:", X_train_normal_sample.shape)
print("Train class distribution:")
print(y_train.value_counts())
print("Test class distribution:")
print(y_test.value_counts())

In [ ]:
# Coarse tuning of nu
nu_values = [0.001, 0.003, 0.005, 0.01, 0.02, 0.05]
nu_rows = []
for nu in nu_values:
    model = OneClassSVM(kernel="rbf", nu=nu, gamma="scale")
    model.fit(X_train_normal_sample)
    pred = np.where(model.predict(X_test) == -1, 1, 0)
    nu_rows.append({
        "nu": nu,
        "anomaly_count": np.sum(pred == 1),
        "fraud_detected": np.sum((y_test.values == 1) & (pred == 1)),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0)
    })
ocsvm_nu_results = pd.DataFrame(nu_rows)
ocsvm_nu_results.sort_values(by="f1_score", ascending=False)

In [ ]:
# Tuning gamma around the best nu
best_nu = 0.005
gamma_values = [0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008]
gamma_rows = []
for gamma in gamma_values:
    model = OneClassSVM(kernel="rbf", nu=best_nu, gamma=gamma)
    model.fit(X_train_normal_sample)
    pred = np.where(model.predict(X_test) == -1, 1, 0)
    gamma_rows.append({
        "nu": best_nu,
        "gamma": gamma,
        "anomaly_count": np.sum(pred == 1),
        "fraud_detected": np.sum((y_test.values == 1) & (pred == 1)),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0)
    })
ocsvm_gamma_results = pd.DataFrame(gamma_rows)
ocsvm_gamma_results.sort_values(by="f1_score", ascending=False)

In [ ]:
final_ocsvm = OneClassSVM(kernel="rbf", nu=0.005, gamma=0.004)
final_ocsvm.fit(X_train_normal_sample)
final_ocsvm_pred = np.where(final_ocsvm.predict(X_test) == -1, 1, 0)

print("Final One-Class SVM: kernel=rbf, nu=0.005, gamma=0.004")
print("Prediction counts:")
print(pd.Series(final_ocsvm_pred).value_counts())
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_ocsvm_pred))
print("Classification Report:")
print(classification_report(y_test, final_ocsvm_pred, target_names=["Normal", "Fraud"]))

## 8. Autoencoder with TensorFlow/Keras

The Autoencoder is trained only on normal transactions. Reconstruction error is used as the anomaly score. A threshold is selected from normal training reconstruction errors.

In [ ]:
X_train_auto = X_train_normal.values.astype("float32")
X_test_auto = X_test.values.astype("float32")
y_test_auto = y_test.values

print("Autoencoder training data shape:", X_train_auto.shape)
print("Autoencoder test data shape:", X_test_auto.shape)
print("Test labels shape:", y_test_auto.shape)
print("Test class distribution:")
print(pd.Series(y_test_auto).value_counts())

In [ ]:
input_dim = X_train_auto.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation="relu")(input_layer)
encoded = Dense(8, activation="relu")(encoded)
latent = Dense(4, activation="relu")(encoded)
decoded = Dense(8, activation="relu")(latent)
decoded = Dense(16, activation="relu")(decoded)
output_layer = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = autoencoder.fit(
    X_train_auto,
    X_train_auto,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    shuffle=True,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
X_test_reconstructed = autoencoder.predict(X_test_auto)
reconstruction_errors = np.mean(np.square(X_test_auto - X_test_reconstructed), axis=1)

autoencoder_results = pd.DataFrame({
    "Actual_Class": y_test_auto,
    "Reconstruction_Error": reconstruction_errors
})

autoencoder_results.groupby("Actual_Class")["Reconstruction_Error"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(autoencoder_results[autoencoder_results["Actual_Class"] == 0]["Reconstruction_Error"],
         bins=100, alpha=0.6, label="Normal")
plt.hist(autoencoder_results[autoencoder_results["Actual_Class"] == 1]["Reconstruction_Error"],
         bins=100, alpha=0.8, label="Fraud")
plt.xlim(0, 50)
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.title("Zoomed Autoencoder Reconstruction Error Distribution")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
X_train_reconstructed = autoencoder.predict(X_train_auto)
train_reconstruction_errors = np.mean(np.square(X_train_auto - X_train_reconstructed), axis=1)

threshold_percentiles = [95, 97, 98, 99, 99.5, 99.7, 99.9]
threshold_rows = []
for p in threshold_percentiles:
    threshold = np.percentile(train_reconstruction_errors, p)
    pred = (autoencoder_results["Reconstruction_Error"] > threshold).astype(int)
    threshold_rows.append({
        "percentile": p,
        "threshold": threshold,
        "anomaly_count": np.sum(pred == 1),
        "fraud_detected": np.sum((y_test_auto == 1) & (pred == 1)),
        "precision": precision_score(y_test_auto, pred, zero_division=0),
        "recall": recall_score(y_test_auto, pred, zero_division=0),
        "f1_score": f1_score(y_test_auto, pred, zero_division=0)
    })
autoencoder_threshold_results = pd.DataFrame(threshold_rows)
autoencoder_threshold_results.sort_values(by="f1_score", ascending=False)

In [ ]:
final_threshold_percentile = 99.7
final_autoencoder_threshold = np.percentile(train_reconstruction_errors, final_threshold_percentile)
final_autoencoder_pred = (autoencoder_results["Reconstruction_Error"] > final_autoencoder_threshold).astype(int)

print("Final Autoencoder threshold percentile:", final_threshold_percentile)
print("Threshold:", final_autoencoder_threshold)
print("Prediction counts:")
print(pd.Series(final_autoencoder_pred).value_counts())
print("Confusion Matrix:")
print(confusion_matrix(y_test_auto, final_autoencoder_pred))
print("Classification Report:")
print(classification_report(y_test_auto, final_autoencoder_pred, target_names=["Normal", "Fraud"]))

In [ ]:
encoder = Model(inputs=input_layer, outputs=latent)
X_test_latent = encoder.predict(X_test_auto)

latent_df = pd.DataFrame(X_test_latent, columns=["Latent_1", "Latent_2", "Latent_3", "Latent_4"])
latent_df["Class"] = y_test_auto

print("Latent representation shape:", latent_df.shape)
latent_df.head()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(latent_df[latent_df["Class"] == 0]["Latent_1"],
            latent_df[latent_df["Class"] == 0]["Latent_2"],
            alpha=0.3, s=8, label="Normal")

plt.scatter(latent_df[latent_df["Class"] == 1]["Latent_1"],
            latent_df[latent_df["Class"] == 1]["Latent_2"],
            alpha=0.9, s=25, label="Fraud")

plt.xlim(0, 100)
plt.ylim(0, 150)
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.title("Zoomed Autoencoder Latent Space Visualization")
plt.legend()
plt.grid(True)
plt.show()

## 9. Model Comparison

This section compares the tuned models. DBSCAN was evaluated on the t-SNE subset, while One-Class SVM and Autoencoder were evaluated on the full test set. This distinction is important when interpreting the results.

In [ ]:
ocsvm_scores = -final_ocsvm.decision_function(X_test)
autoencoder_scores = autoencoder_results["Reconstruction_Error"].values

final_comparison = pd.DataFrame([
    {
        "Model": "DBSCAN",
        "Data Used": "t-SNE subset",
        "Main Parameters": "eps=6, min_samples=20",
        "Accuracy": accuracy_score(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"]),
        "Precision": precision_score(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"], zero_division=0),
        "Recall": recall_score(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"], zero_division=0),
        "F1-score": f1_score(final_dbscan_results["Actual_Class"], final_dbscan_results["Predicted_Anomaly"], zero_division=0),
        "ROC-AUC": np.nan,
        "PR-AUC": np.nan
    },
    {
        "Model": "One-Class SVM",
        "Data Used": "Full test set",
        "Main Parameters": "kernel=rbf, nu=0.005, gamma=0.004",
        "Accuracy": accuracy_score(y_test, final_ocsvm_pred),
        "Precision": precision_score(y_test, final_ocsvm_pred, zero_division=0),
        "Recall": recall_score(y_test, final_ocsvm_pred, zero_division=0),
        "F1-score": f1_score(y_test, final_ocsvm_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, ocsvm_scores),
        "PR-AUC": average_precision_score(y_test, ocsvm_scores)
    },
    {
        "Model": "Autoencoder",
        "Data Used": "Full test set",
        "Main Parameters": "threshold=99.7 percentile",
        "Accuracy": accuracy_score(y_test_auto, final_autoencoder_pred),
        "Precision": precision_score(y_test_auto, final_autoencoder_pred, zero_division=0),
        "Recall": recall_score(y_test_auto, final_autoencoder_pred, zero_division=0),
        "F1-score": f1_score(y_test_auto, final_autoencoder_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test_auto, autoencoder_scores),
        "PR-AUC": average_precision_score(y_test_auto, autoencoder_scores)
    }
])

final_comparison_rounded = final_comparison.copy()
for col in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]:
    final_comparison_rounded[col] = final_comparison_rounded[col].round(3)
final_comparison_rounded

In [ ]:
comparison_plot = final_comparison.copy()
metrics = ["Precision", "Recall", "F1-score"]
comparison_plot.set_index("Model")[metrics].plot(kind="bar", figsize=(10, 6))
plt.title("Final Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.legend(title="Metric")
plt.show()

In [ ]:
fpr_ocsvm, tpr_ocsvm, _ = roc_curve(y_test, ocsvm_scores)
fpr_auto, tpr_auto, _ = roc_curve(y_test_auto, autoencoder_scores)

plt.figure(figsize=(10, 6))
plt.plot(fpr_ocsvm, tpr_ocsvm, label=f"One-Class SVM (AUC = {auc(fpr_ocsvm, tpr_ocsvm):.3f})")
plt.plot(fpr_auto, tpr_auto, label=f"Autoencoder (AUC = {auc(fpr_auto, tpr_auto):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: One-Class SVM vs Autoencoder")
plt.legend()
plt.grid(True)
plt.show()

precision_ocsvm, recall_ocsvm, _ = precision_recall_curve(y_test, ocsvm_scores)
precision_auto, recall_auto, _ = precision_recall_curve(y_test_auto, autoencoder_scores)

plt.figure(figsize=(10, 6))

plt.plot(recall_ocsvm, precision_ocsvm,
         label=f"One-Class SVM (PR-AUC = {average_precision_score(y_test, ocsvm_scores):.3f})")

plt.plot(recall_auto, precision_auto,
         label=f"Autoencoder (PR-AUC = {average_precision_score(y_test_auto, autoencoder_scores):.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve: One-Class SVM vs Autoencoder")
plt.legend()
plt.grid(True)
plt.show()

## 10. Final Conclusion

DBSCAN was useful for density-based clustering and noise detection, and it achieved the highest F1-score on the visualization subset. However, it was not evaluated on the same full test set as the other models, so it is best interpreted as an exploratory clustering/anomaly method.

One-Class SVM achieved the highest fraud recall on the full test set, so it is useful when the priority is catching as many fraud cases as possible, even with more false positives.

The Autoencoder achieved the strongest full-test-set ranking performance, with the highest ROC-AUC and PR-AUC among the full-test-set models. It also provided learned latent representations and reconstruction errors, satisfying the deep learning requirements of the project.